### Setup & Imports

In [1]:
import pandas as pd
import ast
from langchain_community.vectorstores import FAISS
import voyageai

### Load Recipes + FAISS

In [2]:
df = pd.read_csv("final_recipes_data_clean.csv")
df.set_index("id", inplace=True)

vo = voyageai.Client()
vector_store = FAISS.load_local("faiss_index", None, allow_dangerous_deserialization=True)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


### Fridge Inventory (Manual Input for Now)

In [3]:
fridge_inventory = ["chicken", "tomato", "onion", "garlic"]

### Search Function with Inventory Filtering

In [4]:
def search_recipes(query, inventory, k=10):
    query_vec = vo.embed([query], model="voyage-3.5").embeddings
    results = vector_store.similarity_search_by_vector(query_vec[0], k=k)

    scored = []
    for r in results:
        rid = int(r.metadata["recipe_id"])
        if rid not in df.index:
            continue

        recipe = df.loc[rid]
        try:
            ingredients = ast.literal_eval(recipe["ingredients_cleaned"])
        except:
            ingredients = recipe["ingredients_cleaned"]

        if isinstance(ingredients, list):
            available = [item for item in ingredients if any(x.lower() in item.lower() for x in inventory)]
            coverage = len(available) / len(ingredients)
        else:
            coverage = 0

        scored.append((recipe, coverage))

    # Sort descending by coverage
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


### Display Recipes in a DataFrame

In [8]:
def display_recipes(recipes):
    data = []
    for recipe, coverage in recipes:
        try:
            ingredients = ast.literal_eval(recipe["ingredients_cleaned"])
        except:
            ingredients = recipe["ingredients_cleaned"]

        data.append({
            "Recipe Name": recipe["name"],
            "Servings": recipe.get("servings", "N/A"),
            "Coverage": f"{coverage:.0%}",
            "Ingredients": ", ".join(ingredients) if isinstance(ingredients, list) else ingredients
        })

    return pd.DataFrame(data)


### Test Query

In [9]:
user_inventory = input("Enter the ingredients you have (comma separated): ").split(",")
user_inventory = [x.strip() for x in user_inventory if x.strip()]

user_query = input("Enter what type of recipe you want: ")

recipes = search_recipes(user_query, user_inventory, k=10)
df_recipes = display_recipes(recipes)
print(df_recipes)

                   Recipe Name  Servings Coverage  \
0  Chicken or Turkey a La King       6.0      20%   
1             Chicken Ala King       3.0      20%   
2         Chicken a La Charlie       6.0      20%   
3             Cherried Chicken       4.0      20%   
4         Corn-Crisped Chicken       4.0      17%   
5            Chicken Fantastic       6.0      14%   
6       Crunchy Chicken Pieces       4.0      14%   
7                Amish Chicken       2.0      12%   
8                 Chicken King       6.0       8%   
9           Brown Stew Chicken       6.0       8%   

                                         Ingredients  
0  ½ cup chopped green bell pepper, 1 (4 oz) can ...  
1  ¼ cup butter, ⅓ cup flour, ½ teaspoon salt, ¼ ...  
2  8 to 10 chicken pieces, 1 teaspoon garlic powd...  
3  2 & ½ to 3 lbs chicken drumsticks, skinned, 1 ...  
4  2 & ½ to 3 lbs frying chicken, cut up, 2 cups ...  
5  1 roasting chickens, cut up or 6 bone in chick...  
6  1 kg chicken thigh fillet, c